# Aula 02: Limpeza de Texto, Funções Customizadas e Cruzamento Lógico
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

Nesta segunda aula, nosso foco é tratar as **6 variáveis categóricas (texto)** da base:
`setor`, `produto`, `vendedor`, `regiao`, `estado` e `cliente`.

O que vamos aprender:
1. Como inspecionar categorias sujas usando `.value_counts()`.
2. Como criar uma função em Python (`def`) usando `difflib.get_close_matches` para corrigir erros de digitação automaticamente com `.apply()`.
3. Como recuperar valores nulos em `setor` e `regiao` cruzando informações que já estão na linha com `.map()` e `.fillna()`.

In [ ]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# Recarregando e aplicando os passos rápidos da Aula 01
df = pd.read_excel('Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)
print('Base carregada da Aula 01:', df.shape)

## 1. Investigando a Sujeira nas Colunas de Texto
Quando olhamos para `.value_counts()` de qualquer coluna de texto, percebemos um padrão claro: as categorias corretas aparecem dezenas ou centenas de vezes, enquanto os erros de digitação (ex: `"Cvrnes"`, `"Vacessa"`, `"tarejo"`) aparecem de 1 a 4 vezes.

In [ ]:
colunas_texto = ['setor', 'produto', 'vendedor', 'regiao', 'estado', 'cliente']

for col in colunas_texto:
    print(f'=== Coluna: {col.upper()} (Total de valores únicos: {df[col].nunique()}) ===')
    print(df[col].value_counts().head(15))
    print('\n')

## 2. Automatizando a Correção de Erros de Digitação (`difflib`)
Em vez de fazer mais de 800 `.replace()` manualmente, podemos listar apenas os nomes verdadeiros de cada categoria (aqueles com mais de 10 ocorrências) e criar uma função que compara cada palavra suja com a lista oficial, substituindo pela palavra mais parecida!

In [ ]:
# Dicionário com as categorias oficiais (verdadeiras) de cada coluna
categorias_oficiais = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': [
        'Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 
        'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'
    ],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': [
        'Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 
        'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'
    ],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}

def corrigir_palavra(valor, lista_oficial):
    """
    Recebe um valor da célula e retorna a palavra mais próxima da lista_oficial
    se a semelhança for de pelo menos 60% (cutoff=0.6).
    """
    if pd.isna(valor):
        return np.nan
    
    palavra_limpa = str(valor).strip()
    match = get_close_matches(palavra_limpa, lista_oficial, n=1, cutoff=0.6)
    
    return match[0] if match else np.nan

# Teste rápido da função antes de aplicar na tabela inteira:
print('Teste 1 ("Cvrnes") ->', corrigir_palavra('Cvrnes', categorias_oficiais['setor']))
print('Teste 2 ("Vacessa") ->', corrigir_palavra('Vacessa', categorias_oficiais['vendedor']))
print('Teste 3 ("Attcarejo") ->', corrigir_palavra('Attcarejo', categorias_oficiais['cliente']))

In [ ]:
# Aplicando a nossa função nas 6 colunas de texto usando um loop for
for col, lista_valida in categorias_oficiais.items():
    df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))
    print(f'Coluna "{col}" limpa! Valores únicos agora:', df[col].unique().tolist())

## 3. Recuperação Inteligente de Valores Nulos (Cruzamento Lógico)
Mesmo após corrigir os erros de digitação, ainda temos valores vazios (`NaN`) em todas as colunas.
Porém, pense como um Cientista de Dados:
* Se o `setor` está vazio, mas sabemos que o `produto` daquela linha é **Sorvete**, qual é o setor? **Congelados**!
* Se a `regiao` está vazia, mas sabemos que o `estado` é **Minas Gerais**, qual é a região? **Sudeste**!

In [ ]:
print('Nulos ANTES do cruzamento lógico:')
print(df[['setor', 'produto', 'regiao', 'estado']].isna().sum())

# Criando os dicionários de relacionamento
mapa_produto_setor = {
    'Queijo': 'Latícinio', 'Leite': 'Latícinio', 'Coalhada': 'Latícinio',
    'Sorvete': 'Congelados', 'Lasanha': 'Congelados', 'Pizza': 'Congelados',
    'Frango': 'Carnes', 'Bovina': 'Carnes', 'Suína': 'Carnes',
    'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'
}

mapa_estado_regiao = {
    'Rio de Janeiro': 'Sudeste', 'Minas Gerais': 'Sudeste', 'São Paulo': 'Sudeste',
    'Bahia': 'Nordeste', 'Rio Grande do Norte': 'Nordeste', 'Sergipe': 'Nordeste',
    'Rio Grande do Sul': 'Sul', 'Paraná': 'Sul', 'Santa Catarina': 'Sul'
}

# Preenchendo os nulos usando .fillna() + .map()
df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))

print('\nNulos DEPOIS do cruzamento lógico:')
print(df[['setor', 'produto', 'regiao', 'estado']].isna().sum())

Recuperamos **81 setores** e **78 regiões** sem precisar chutar nenhum dado — apenas usando a regra de negócio da própria tabela!